In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if cwd.name == 'url_corpus' and cwd.parent.name == 'notebooks':
    os.chdir(cwd.parent)

nb_dir = Path.cwd().resolve()
backend_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir
if str(backend_root) not in sys.path:
    sys.path.insert(0, str(backend_root))

print(f'Notebook CWD: {Path.cwd()}')
print(f'Backend root in sys.path: {backend_root}')


# Detekcja Phishingu - XGBoost

Ten notebook zawiera implementację i analizę modelu XGBoost do wykrywania phishingu.

## 1. Import bibliotek i konfiguracja


In [ ]:
import sys
cwd = Path.cwd().resolve(); os.chdir(cwd.parent) if cwd.name == 'url_corpus' and cwd.parent.name == 'notebooks' else None; nb_dir = Path.cwd().resolve(); backend_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir; sys.path.insert(0, str(backend_root)) if str(backend_root) not in sys.path else None

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import pickle

import xgboost as xgb
import importlib

if 'utils.resource_monitor' in sys.modules:
    importlib.reload(sys.modules['utils.resource_monitor'])

from utils.resource_monitor import ResourceMonitor
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, learning_curve, validation_curve
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                           roc_auc_score, confusion_matrix, roc_curve)
from utils.feature_names import get_readable_name

import os

PLOTS_DIR = "../static/plots_url_corpus/xgboost"
MODELS_DIR = "../models/xgboost"
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.titlesize'] = 16
plt.rcParams['figure.figsize'] = (12, 7)

np.random.seed(42)

print("Biblioteki zostały zaimportowane pomyślnie!")


## 2. Wczytanie i eksploracja danych (oba zbiory)


In [ ]:
df_original = pd.read_csv('../data/url_corpus/dataset_full.csv')
df_optimized = pd.read_csv('../data/url_corpus/dataset_optimized.csv')

datasets = {
    'Pełny zbiór': df_original,
    'Zoptymalizowany zbiór': df_optimized
}

print("="*60)
print("PORÓWNANIE ZBIORÓW DANYCH")
print("="*60)

for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"Zbiór: {name.upper()}")
    print(f"{'='*60}")
    print(f"Rozmiar zbioru: {df.shape}")
    print(f"Liczba cech: {len(df.columns)-1}")

    print(f"\nPierwsze 5 kolumn: {list(df.columns[:5])}")
    print(f"Ostatnie 5 kolumn: {list(df.columns[-5:])}")

    label_counts = df['phishing'].value_counts()
    print(f"\nRozkład etykiet:")
    print(f"URL oznaczone jako bezpieczne: {label_counts[0]:,} ({label_counts[0]/len(df)*100:.1f}%)")
    print(f"URL oznaczone jako phisingowe: {label_counts[1]:,} ({label_counts[1]/len(df)*100:.1f}%)")

    print(f"\nInformacje o zbiorze:")
    print(f"- Łączna liczba próbek: {len(df):,}")
    print(f"- Liczba cech: {len(df.columns)-1}")
    print(f"- Brakujące wartości: {df.isnull().sum().sum()}")
    print(f"- Duplikaty: {df.duplicated().sum()}")


## 3. Przygotowanie cech i etykiet (dla obu zbiorów)


In [ ]:
data_splits = {}

for name, df in datasets.items():
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]

    print(f"\n{'='*60}")
    print(f"Przygotowanie danych: {name.upper()}")
    print(f"{'='*60}")
    print(f"  Liczba cech: {X.shape[1]}")
    print(f"  Liczba próbek: {X.shape[0]:,}")

    print(f"\nRozkład klas:")
    for label, count in y.value_counts().items():
        percentage = count / len(y) * 100
        label_name = "Brak Phishingu" if label == 0 else "Phishing"
        print(f"  {label} ({label_name}): {count:,} ({percentage:.1f}%)")

    class_ratio = y.value_counts().min() / y.value_counts().max()
    scale_pos_weight = y.value_counts()[0] / y.value_counts()[1]
    print(f"\nObliczony scale_pos_weight dla XGBoost: {scale_pos_weight:.2f}")

    data_splits[name] = {
        'X': X,
        'y': y,
        'scale_pos_weight': scale_pos_weight
    }


## 4. Podział danych na zbiory uczący i testowy (dla obu zbiorów)


In [ ]:
for name in data_splits.keys():
    X = data_splits[name]['X']
    y = data_splits[name]['y']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    data_splits[name]['X_train'] = X_train
    data_splits[name]['X_test'] = X_test
    data_splits[name]['y_train'] = y_train
    data_splits[name]['y_test'] = y_test

    print(f"\n{'='*60}")
    print(f"Podział danych: {name.upper()}")
    print(f"{'='*60}")
    print(f"Zbiór uczący: {X_train.shape[0]:,} próbek ({X_train.shape[0]/len(X)*100:.1f}%)")
    print(f"Zbiór testowy: {X_test.shape[0]:,} próbek ({X_test.shape[0]/len(X)*100:.1f}%)")

    print(f"\nRozkład klas w zbiorze uczącym:")
    train_counts = y_train.value_counts()
    print(f"  Brak phishingu: {train_counts[0]:,} ({train_counts[0]/len(y_train)*100:.1f}%)")
    print(f"  Phishing: {train_counts[1]:,} ({train_counts[1]/len(y_train)*100:.1f}%)")

    print(f"\nRozkład klas w zbiorze testowym:")
    test_counts = y_test.value_counts()
    print(f"  Brak phishingu: {test_counts[0]:,} ({test_counts[0]/len(y_test)*100:.1f}%)")
    print(f"  Phishing: {test_counts[1]:,} ({test_counts[1]/len(y_test)*100:.1f}%)")

## 5. Optymalizacja hiperparametrów (dla obu zbiorów)


In [ ]:
optimization_results = {}

for dataset_name in ['Pełny zbiór', 'Zoptymalizowany zbiór']:
    print(f"\n{'='*80}")
    print(f"OPTYMALIZACJA HIPERPARAMETRÓW - {dataset_name.upper()}")
    print(f"{'='*80}")

    X_train = data_splits[dataset_name]['X_train']
    X_test = data_splits[dataset_name]['X_test']
    y_train = data_splits[dataset_name]['y_train']
    y_test = data_splits[dataset_name]['y_test']
    scale_pos_weight = data_splits[dataset_name]['scale_pos_weight']

    print(f"Stosunek klas (negatywne/pozytywne): {scale_pos_weight:.2f}")

    param_grids = [
        {
            'name': 'Optymalizacja learning_rate i n_estimators',
            'params': {
                'learning_rate': [0.01, 0.1, 0.3],
                'n_estimators': [100, 200, 300],
                'max_depth': [6],
                'min_child_weight': [1],
                'subsample': [1.0],
                'colsample_bytree': [1.0],
                'scale_pos_weight': [scale_pos_weight]
            }
        },
        {
            'name': 'Optymalizacja max_depth i min_child_weight',
            'params': {
                'learning_rate': [0.1],
                'n_estimators': [200],
                'max_depth': [3, 6, 9],
                'min_child_weight': [1, 3, 5],
                'subsample': [1.0],
                'colsample_bytree': [1.0],
                'scale_pos_weight': [scale_pos_weight]
            }
        },
        {
            'name': 'Optymalizacja subsample i colsample_bytree',
            'params': {
                'learning_rate': [0.1],
                'n_estimators': [200],
                'max_depth': [6],
                'min_child_weight': [1],
                'subsample': [0.6, 0.8, 1.0],
                'colsample_bytree': [0.6, 0.8, 1.0],
                'scale_pos_weight': [scale_pos_weight]
            }
        }
    ]

    best_overall_score = 0
    best_overall_params = None
    best_grid_name = ""

    start_time = time.time()

    for i, grid_config in enumerate(param_grids):
        grid_name = grid_config['name']
        param_grid = grid_config['params']

        combinations = (len(param_grid['learning_rate']) *
                       len(param_grid['n_estimators']) *
                       len(param_grid['max_depth']) *
                       len(param_grid['min_child_weight']) *
                       len(param_grid['subsample']) *
                       len(param_grid['colsample_bytree']))

        print(f"\n{'='*60}")
        print(f"Konfiguracja {i+1}/{len(param_grids)}: {grid_name}")
        print(f"{'='*60}")
        print(f"Testowanie {combinations} kombinacji parametrów...")

        grid_start_time = time.time()

        grid_search = GridSearchCV(
            xgb.XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss', use_label_encoder=False),
            param_grid,
            cv=5,
            scoring='roc_auc',
            n_jobs=-1,
            verbose=0
        )

        grid_search.fit(X_train, y_train)

        grid_time = time.time() - grid_start_time

        print(f"\nNajlepsze parametry: {grid_search.best_params_}")
        print(f"Najlepszy CV ROC-AUC: {grid_search.best_score_:.4f}")
        print(f"Czas wykonania: {grid_time:.2f} sekund")

        if grid_search.best_score_ > best_overall_score:
            best_overall_score = grid_search.best_score_
            best_overall_params = grid_search.best_params_
            best_grid_name = grid_name

    optimization_time = time.time() - start_time

    print(f"\n{'='*60}")
    print(f"PODSUMOWANIE OPTYMALIZACJI - {dataset_name.upper()}")
    print(f"{'='*60}")
    print(f"Całkowity czas optymalizacji: {optimization_time:.2f} sekund ({optimization_time/60:.1f} minut)")
    print(f"Najlepszy CV ROC-AUC: {best_overall_score:.4f}")
    print(f"Najlepsza konfiguracja: {best_grid_name}")
    print(f"Najlepsze parametry:")
    for param, value in best_overall_params.items():
        print(f"   {param}: {value}")

    if best_overall_score >= 0.95:
        print("\n✅ Doskonała wydajność CV (≥95%)")
    elif best_overall_score >= 0.90:
        print("\n✅ Bardzo dobra wydajność CV (≥90%)")
    elif best_overall_score >= 0.85:
        print("\n✅ Dobra wydajność CV (≥85%)")
    else:
        print("\n⚠️ Wydajność CV wymaga poprawy (<85%)")

    optimization_results[dataset_name] = {
        'best_params': best_overall_params,
        'best_score': best_overall_score,
        'best_grid_name': best_grid_name,
        'optimization_time': optimization_time,
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }

print(f"\n{'='*80}")
print("ZAKOŃCZONO OPTYMALIZACJĘ DLA OBU ZBIORÓW")
print(f"{'='*80}")

## 5.1. Trening i monitorowanie zasobów systemowych (RAM i CPU)


In [ ]:
results_for_datasets = {}

for dataset_name in ['Pełny zbiór', 'Zoptymalizowany zbiór']:
    print(f"\n{'='*80}")
    print(f"TRENING Z MONITOROWANIEM ZASOBÓW - {dataset_name.upper()}")
    print(f"{'='*80}")

    X_train = optimization_results[dataset_name]['X_train']
    X_test = optimization_results[dataset_name]['X_test']
    y_train = optimization_results[dataset_name]['y_train']
    y_test = optimization_results[dataset_name]['y_test']
    best_params = optimization_results[dataset_name]['best_params']

    print(f"Trenowanie na pełnym zbiorze uczącym: {X_train.shape[0]:,} próbek, {X_train.shape[1]} cech")
    print(f"Parametry modelu: {best_params}")

    resource_monitor = ResourceMonitor(interval=0.1, debug=False)
    resource_monitor.start()

    training_start = time.time()
    best_model = xgb.XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss', use_label_encoder=False, **best_params)
    best_model.fit(X_train, y_train)
    training_time = time.time() - training_start

    resource_monitor.stop()

    stats = resource_monitor.get_stats()

    results_for_datasets[dataset_name] = {
        'model': best_model,
        'best_params': best_params,
        'best_score': optimization_results[dataset_name]['best_score'],
        'best_grid_name': optimization_results[dataset_name]['best_grid_name'],
        'optimization_time': optimization_results[dataset_name]['optimization_time'],
        'training_time': training_time,
        'resource_monitor': resource_monitor,
        'resource_stats': stats,
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }

    if stats:
        print(f"\nWYNIKI MONITOROWANIA:")
        print(f"  • Czas treningu: {training_time:.2f}s")
        print(f"  • Liczba próbek monitorowania: {stats['samples']}")
        print(f"  • CPU:")
        print(f"    - Min: {stats['cpu']['min']:.1f}%")
        print(f"    - Max: {stats['cpu']['max']:.1f}%")
        print(f"    - Średnia: {stats['cpu']['mean']:.1f}%")
        print(f"  • RAM:")
        print(f"    - Min: {stats['ram']['min']:.2f} GB")
        print(f"    - Max: {stats['ram']['max']:.2f} GB")
        print(f"    - Średnia: {stats['ram']['mean']:.2f} GB")

print(f"\n{'='*80}")
print("ZAKOŃCZONO TRENOWANIE MODELI DLA OBU ZBIORÓW")
print(f"{'='*80}")


## 5.1. Monitorowanie zasobów systemowych (RAM i CPU)


In [ ]:
print(f"\n{'='*80}")
print("SZCZEGÓŁOWE STATYSTYKI WYKORZYSTANIA ZASOBÓW")
print(f"{'='*80}")

dataset_display_names = {
    'Pełny zbiór': 'Pełny zbiór (111 cech)',
    'Zoptymalizowany zbiór': 'Zoptymalizowany zbiór (73 cechy)'
}

for dataset_name in ['Pełny zbiór', 'Zoptymalizowany zbiór']:
    stats = results_for_datasets[dataset_name]['resource_stats']
    training_time = results_for_datasets[dataset_name]['training_time']
    display_name = dataset_display_names[dataset_name]

    if stats:
        print(f"\n{display_name}:")
        print(f"  • Czas treningu: {training_time:.2f}s")
        print(f"  • Liczba próbek: {stats['samples']}")
        print(f"  • CPU:")
        print(f"    - Min: {stats['cpu']['min']:.1f}%")
        print(f"    - Max: {stats['cpu']['max']:.1f}%")
        print(f"    - Średnia: {stats['cpu']['mean']:.1f}%")
        print(f"  • RAM:")
        print(f"    - Min: {stats['ram']['min']:.2f} GB")
        print(f"    - Max: {stats['ram']['max']:.2f} GB")
        print(f"    - Średnia: {stats['ram']['mean']:.2f} GB")

## 5.2. Wizualizacja wykorzystania zasobów systemowych


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Monitorowanie zasobów systemowych podczas uczenia modeli XGBoost', fontsize=18, fontweight='bold')

dataset_names = ['Pełny zbiór', 'Zoptymalizowany zbiór']
dataset_display_names_list = ['Pełny zbiór (111 cech)', 'Zoptymalizowany zbiór (73 cechy)']
colors = ['#1f77b4', '#ff7f0e']

for idx, dataset_name in enumerate(dataset_names):
    monitor = results_for_datasets[dataset_name]['resource_monitor']
    stats = results_for_datasets[dataset_name]['resource_stats']
    display_name = dataset_display_names_list[idx]

    if stats and len(monitor.timestamps) > 0:
        time_array = np.array(monitor.timestamps)
        cpu_array = np.array(monitor.cpu_percent)
        ram_array = np.array(monitor.ram_gb)

        time_array = time_array - time_array[0]

        axes[0, idx].plot(time_array, cpu_array, color=colors[idx], linewidth=2, alpha=0.7)
        axes[0, idx].axhline(y=stats['cpu']['mean'], color='red', linestyle='--',
                            linewidth=2, label=f'Średnia: {stats["cpu"]["mean"]:.1f}%')
        axes[0, idx].fill_between(time_array, cpu_array, alpha=0.3, color=colors[idx])
        axes[0, idx].set_xlabel('Czas [s]', fontsize=12, fontweight='bold')
        axes[0, idx].set_ylabel('Wykorzystanie CPU [%]', fontsize=12, fontweight='bold')
        axes[0, idx].set_title(f'CPU - {display_name}', fontweight='bold', fontsize=14)
        axes[0, idx].legend(fontsize=10)
        axes[0, idx].grid(True, alpha=0.3)
        axes[0, idx].set_ylim([0, max(20, stats['cpu']['max'] * 1.1)])
        axes[0, idx].set_xlim([0, time_array[-1]])

        axes[1, idx].plot(time_array, ram_array, color=colors[idx], linewidth=2, alpha=0.7)
        axes[1, idx].axhline(y=stats['ram']['mean'], color='red', linestyle='--',
                            linewidth=2, label=f'Średnia: {stats["ram"]["mean"]:.2f} GB')
        axes[1, idx].fill_between(time_array, ram_array, alpha=0.3, color=colors[idx])
        axes[1, idx].set_xlabel('Czas [s]', fontsize=12, fontweight='bold')
        axes[1, idx].set_ylabel('Wykorzystanie RAM [GB]', fontsize=12, fontweight='bold')
        axes[1, idx].set_title(f'RAM - {display_name}', fontweight='bold', fontsize=14)
        axes[1, idx].legend(fontsize=10)
        axes[1, idx].grid(True, alpha=0.3)
        axes[1, idx].set_ylim([0, stats['ram']['max'] * 1.2])
        axes[1, idx].set_xlim([0, time_array[-1]])

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/xgboost_resource_usage_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Wizualizacja wykorzystania zasobów zapisana")

## 6. Ewaluacja modeli (dla obu zbiorów)


In [ ]:
evaluation_results = {}

for dataset_name in ['Pełny zbiór', 'Zoptymalizowany zbiór']:
    print(f"\n{'='*80}")
    print(f"EWALUACJA MODELU - {dataset_name.upper()}")
    print(f"{'='*80}")

    model = results_for_datasets[dataset_name]['model']
    X_train = results_for_datasets[dataset_name]['X_train']
    X_test = results_for_datasets[dataset_name]['X_test']
    y_train = results_for_datasets[dataset_name]['y_train']
    y_test = results_for_datasets[dataset_name]['y_test']

    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_roc = roc_auc_score(y_test, y_pred_proba)

    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')

    evaluation_results[dataset_name] = {
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'confusion_matrix': cm,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc_roc': auc_roc,
        'cv_scores': cv_scores,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }

    print(f"\nWYNIKI NA ZBIORZE TESTOWYM:")
    print(f"  • Dokładność (Accuracy): {accuracy:.4f}")
    print(f"  • Precyzja (Precision): {precision:.4f}")
    print(f"  • Czułość (Recall): {recall:.4f}")
    print(f"  • Miara F1 (F1-Score): {f1:.4f}")
    print(f"  • AUC-ROC: {auc_roc:.4f}")

    print(f"\nMACIERZ POMYŁEK:")
    print(f"  • Prawdziwie negatywne (TN): {tn:,}")
    print(f"  • Fałszywie pozytywne (FP): {fp:,}")
    print(f"  • Fałszywie negatywne (FN): {fn:,}")
    print(f"  • Prawdziwie pozytywne (TP): {tp:,}")

    print(f"\nPIĘCIOKROTNA WALIDACJA KRZYŻOWA:")
    print(f"  • Średni wynik ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  • Min: {cv_scores.min():.4f}")
    print(f"  • Max: {cv_scores.max():.4f}")

print(f"\n{'='*80}")
print("PORÓWNANIE WYNIKÓW")
print(f"{'='*80}")
print(f"{'Metryka':<20} {'Pełny zbiór':>15} {'Zoptymalizowany zbiór':>20}")
print(f"{'-'*80}")
for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']:
    full_val = evaluation_results['Pełny zbiór'][metric]
    opt_val = evaluation_results['Zoptymalizowany zbiór'][metric]
    diff = opt_val - full_val
    diff_str = f"({diff:+.4f})" if diff != 0 else ""
    print(f"{metric.capitalize():<20} {full_val:>15.4f} {opt_val:>15.4f} {diff_str:>5}")

## 7. Wizualizacja kompleksowa i krzywa uczenia


In [ ]:
dataset_mapping = {
    'Pełny zbiór': 'Pełny zbiór',
    'Zoptymalizowany zbiór': 'Zoptymalizowany zbiór'
}

renamed_trained_models = {dataset_mapping[k]: v for k, v in results_for_datasets.items()}
renamed_evaluation_results = {dataset_mapping[k]: v for k, v in evaluation_results.items()}
renamed_train_test_data = {
    'Pełny zbiór': {
        'X_train': results_for_datasets['Pełny zbiór']['X_train'],
        'X_test': results_for_datasets['Pełny zbiór']['X_test'],
        'y_test': results_for_datasets['Pełny zbiór']['y_test']
    },
    'Zoptymalizowany zbiór': {
        'X_train': results_for_datasets['Zoptymalizowany zbiór']['X_train'],
        'X_test': results_for_datasets['Zoptymalizowany zbiór']['X_test'],
        'y_test': results_for_datasets['Zoptymalizowany zbiór']['y_test']
    }
}

fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('Porównanie Modeli: Pełny zbiór (111 cech) vs Zoptymalizowany zbiór (73 cechy)',
             fontsize=20, fontweight='bold', y=0.995)

dataset_names = list(renamed_trained_models.keys())
colors = ['#1f77b4', '#ff7f0e']

for idx, dataset_name in enumerate(dataset_names):
    cm = renamed_evaluation_results[dataset_name]['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues' if idx == 0 else 'Oranges',
                xticklabels=['Brak phishingu', 'Phishing'],
                yticklabels=['Brak phishingu', 'Phishing'],
                ax=axes[0, idx], annot_kws={"size": 14}, cbar_kws={'label': 'Liczba próbek'})
    axes[0, idx].set_title(f'Macierz pomyłek - {dataset_name}', fontweight='bold', fontsize=14)
    axes[0, idx].set_ylabel('Prawdziwa etykieta')
    axes[0, idx].set_xlabel('Przewidziana etykieta')

for idx, dataset_name in enumerate(dataset_names):
    y_test = renamed_train_test_data[dataset_name]['y_test']
    y_pred_proba = renamed_evaluation_results[dataset_name]['y_pred_proba']
    auc_roc = renamed_evaluation_results[dataset_name]['auc_roc']

    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    axes[1, idx].plot(fpr, tpr, color=colors[idx], lw=3, label=f'ROC-AUC = {auc_roc:.4f}')
    axes[1, idx].set_xlim([0.0, 1.0])
    axes[1, idx].set_ylim([0.0, 1.05])
    axes[1, idx].set_xlabel('Wskaźnik fałszywie pozytywnych (FPR)', fontsize=12)
    axes[1, idx].set_ylabel('Wskaźnik prawdziwie pozytywnych (TPR)', fontsize=12)
    axes[1, idx].set_title(f'Krzywa ROC - {dataset_name}', fontweight='bold', fontsize=14)
    axes[1, idx].legend(loc="lower right", fontsize=11)
    axes[1, idx].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)

plt.tight_layout(rect=[0, 0, 1, 0.99], w_pad=3.0, h_pad=3.0)
plt.savefig(f'{PLOTS_DIR}/xgboost_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Wizualizacja porównawcza zapisana")

## 7.1. Porównanie metryk w postaci wykresów słupkowych


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

metrics = ['Dokładność', 'Precyzja', 'Czułość', 'F1', 'AUC-ROC']
metric_keys = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
full_values = [renamed_evaluation_results['Pełny zbiór'][key] for key in metric_keys]
opt_values = [renamed_evaluation_results['Zoptymalizowany zbiór'][key] for key in metric_keys]

x = np.arange(len(metrics))
width = 0.35

bars1 = axes[0].bar(x - width/2, full_values, width, label='Pełny zbiór', color='#1f77b4', alpha=0.8)
bars2 = axes[0].bar(x + width/2, opt_values, width, label='Zoptymalizowany zbiór', color='#ff7f0e', alpha=0.8)

axes[0].set_xlabel('Metryki', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Wartość', fontsize=12, fontweight='bold')
axes[0].set_title('Porównanie metryk wydajności', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics, rotation=45, ha='right')
axes[0].legend(fontsize=11)
axes[0].set_ylim([0.7, 1.0])
axes[0].grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.4f}',
                    ha='center', va='bottom', fontsize=9)

labels = ['Czas uczenia (s)', 'Liczba cech']
full_stats = [
    results_for_datasets['Pełny zbiór']['training_time'],
    len(data_splits['Pełny zbiór']['X'].columns)
]
opt_stats = [
    results_for_datasets['Zoptymalizowany zbiór']['training_time'],
    len(data_splits['Zoptymalizowany zbiór']['X'].columns)
]

x_pos = np.arange(len(labels))
bars1 = axes[1].bar(x_pos - width/2, full_stats, width, label='Pełny zbiór', color='#1f77b4', alpha=0.8)
bars2 = axes[1].bar(x_pos + width/2, opt_stats, width, label='Zoptymalizowany zbiór', color='#ff7f0e', alpha=0.8)

axes[1].set_xlabel('Charakterystyka', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Wartość', fontsize=12, fontweight='bold')
axes[1].set_title('Porównanie czasu uczenia i liczby cech', fontsize=14, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(labels)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}',
                    ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/xgboost_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Wizualizacja metryk zapisana")

comparison_df = pd.DataFrame({
    'Zbiór danych': ['Pełny zbiór', 'Zoptymalizowany zbiór'],
    'Dokładność': [renamed_evaluation_results['Pełny zbiór']['accuracy'], renamed_evaluation_results['Zoptymalizowany zbiór']['accuracy']],
    'Precyzja': [renamed_evaluation_results['Pełny zbiór']['precision'], renamed_evaluation_results['Zoptymalizowany zbiór']['precision']],
    'Czułość': [renamed_evaluation_results['Pełny zbiór']['recall'], renamed_evaluation_results['Zoptymalizowany zbiór']['recall']],
    'Miara F1': [renamed_evaluation_results['Pełny zbiór']['f1'], renamed_evaluation_results['Zoptymalizowany zbiór']['f1']],
    'AUC-ROC': [renamed_evaluation_results['Pełny zbiór']['auc_roc'], renamed_evaluation_results['Zoptymalizowany zbiór']['auc_roc']],
    'Średnia CV AUC': [renamed_evaluation_results['Pełny zbiór']['cv_mean'], renamed_evaluation_results['Zoptymalizowany zbiór']['cv_mean']],
    'Odchylenie CV AUC': [renamed_evaluation_results['Pełny zbiór']['cv_std'], renamed_evaluation_results['Zoptymalizowany zbiór']['cv_std']],
    'Czas uczenia (s)': [results_for_datasets['Pełny zbiór']['training_time'], results_for_datasets['Zoptymalizowany zbiór']['training_time']],
    'Liczba cech': [len(data_splits['Pełny zbiór']['X'].columns), len(data_splits['Zoptymalizowany zbiór']['X'].columns)]
})

comparison_df.to_csv('../data/url_corpus/xgboost_comparison.csv', index=False)
print(f"✓ Porównanie zapisane: ../data/url_corpus/xgboost_comparison.csv")
print("\nTabela porównawcza:")
print(comparison_df.to_string(index=False))

## 7.1a. Wizualizacja przykładowego drzewa z modelu XGBoost


In [ ]:
dataset_names_internal = ['Pełny zbiór', 'Zoptymalizowany zbiór']
dataset_display_names = ['Pełny zbiór (111 cech)', 'Zoptymalizowany zbiór (73 cechy)']

for idx, dataset_name in enumerate(dataset_names_internal):
    model = results_for_datasets[dataset_name]['model']
    X_train = results_for_datasets[dataset_name]['X_train']

    fig, ax = plt.subplots(1, 1, figsize=(24, 14))

    booster = model.get_booster()
    num_trees = len(booster.get_dump())

    xgb.plot_tree(model,
                  num_trees=0,
                  ax=ax,
                  rankdir='LR')

    tree_dump = booster.get_dump()[0]
    max_tabs = max(line.count('\t') for line in tree_dump.split('\n') if line.strip())
    actual_depth = max_tabs

    ax.set_title(f'Wizualizacja przykładowego drzewa decyzyjnego z XGBoost - {dataset_display_names[idx]}\n(Pierwsze drzewo z {num_trees} | Głębokość: {actual_depth})',
                fontweight='bold', fontsize=16, pad=15)

    plt.tight_layout()

    filename_suffix = 'Pełny zbiór' if idx == 0 else 'Zoptymalizowany zbiór'
    plt.savefig(f'{PLOTS_DIR}/xgboost_tree_visualization_{filename_suffix}.png', dpi=600, bbox_inches='tight')
    plt.show()

    plt.close(fig)

    print(f"✅ Wizualizacja drzewa dla {dataset_display_names[idx]} zapisana")

print("\n✅ Wszystkie wizualizacje drzew zapisane")

print("\n" + "="*80)
print("STATYSTYKI MODELI XGBOOST")
print("="*80)

for dataset_name, display_name in zip(dataset_names_internal, dataset_display_names):
    model = results_for_datasets[dataset_name]['model']
    booster = model.get_booster()

    num_trees = len(booster.get_dump())
    best_iteration = model.best_iteration if hasattr(model, 'best_iteration') else num_trees

    print(f"\n{display_name}:")
    print(f"  • Liczba drzew (boosting rounds): {num_trees}")
    if hasattr(model, 'best_iteration'):
        print(f"  • Najlepsza iteracja: {best_iteration}")
    print(f"  • Parametry modelu:")
    for param, value in results_for_datasets[dataset_name]['best_params'].items():
        print(f"    - {param}: {value}")

## 7.1b. Analiza predykcji dla przykładowych URL


In [ ]:
dataset_name = 'Zoptymalizowany zbiór'
model = results_for_datasets[dataset_name]['model']
X_test = results_for_datasets[dataset_name]['X_test']
y_test = results_for_datasets[dataset_name]['y_test']

phishing_idx = y_test[y_test == 1].index[0]
safe_idx = y_test[y_test == 0].index[0]

examples = [
    {'idx': phishing_idx, 'label': 'Phishing', 'color': '#ff7f0e'},
    {'idx': safe_idx, 'label': 'Bezpieczny', 'color': '#1f77b4'}
]

print("="*80)
print("ANALIZA PREDYKCJI XGBOOST DLA PRZYKŁADOWYCH URL-I")
print("="*80)

for example in examples:
    idx = example['idx']
    true_label = example['label']
    sample = X_test.loc[idx:idx]

    prediction = model.predict(sample)[0]
    prediction_proba = model.predict_proba(sample)[0]
    predicted_label = 'Phishing' if prediction == 1 else 'Bezpieczny'

    print(f"\n{'='*80}")
    print(f"URL #{idx} - Prawdziwa etykieta: {true_label}")
    print(f"{'='*80}")
    print(f"Predykcja modelu: {predicted_label}")
    print(f"Prawdopodobieństwo:")
    print(f"  • Bezpieczny: {prediction_proba[0]:.4f} ({prediction_proba[0]*100:.2f}%)")
    print(f"  • Phishing: {prediction_proba[1]:.4f} ({prediction_proba[1]*100:.2f}%)")

    print(f"\nTop 10 wartości cech dla tego URL:")
    feature_values = sample.iloc[0]
    feature_names_readable = [get_readable_name(col) for col in X_test.columns]

    feature_importance_scores = model.feature_importances_

    feature_info = list(zip(feature_names_readable, feature_values, feature_importance_scores))
    sorted_features = sorted(feature_info, key=lambda x: abs(x[1]) * x[2], reverse=True)[:10]

    for feat_name, feat_val, feat_imp in sorted_features:
        print(f"  • {feat_name}: {feat_val:.4f} (ważność: {feat_imp:.4f})")
    print(f"\nTop 5 najbardziej wpływowych cech (według ważności modelu):")
    feature_importance_df = pd.DataFrame({
        'Cecha': feature_names_readable,
        'Ważność': feature_importance_scores,
        'Wartość': feature_values
    }).sort_values('Ważność', ascending=False).head(5)

    for _, row in feature_importance_df.iterrows():
        print(f"  • {row['Cecha']}: wartość={row['Wartość']:.4f}, ważność={row['Ważność']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Analiza decyzji modelu XGBoost dla przykładowych URL-i',
             fontsize=16, fontweight='bold', y=1.02)

for idx, example in enumerate(examples):
    sample_idx = example['idx']
    label = example['label']
    sample = X_test.loc[sample_idx:sample_idx]

    prediction_proba = model.predict_proba(sample)[0]

    sizes = [prediction_proba[0] * 100, prediction_proba[1] * 100]
    labels_pie = ['Brak phishingu', 'Phishing']
    explode = (0.05, 0.05)
    colors_pie = ['#2ecc71', '#e74c3c']

    axes[idx].pie(sizes, explode=explode, labels=labels_pie, colors=colors_pie,
                  autopct='%1.1f%%', shadow=True, startangle=90,
                  textprops={'fontsize': 12, 'fontweight': 'bold'})
    axes[idx].set_title(f'URL #{sample_idx}\nPrawdziwa etykieta: {label}',
                       fontweight='bold', fontsize=14, pad=15)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/xgboost_prediction_examples.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Wizualizacja predykcji dla przykładowych URL-i zapisana")

## 7.2. Wpływ rozmiaru zbioru uczącego na skuteczność modelu


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Wpływ rozmiaru zbioru uczącego na skuteczność modelu', fontsize=16, fontweight='bold')

dataset_names = ['Pełny zbiór', 'Zoptymalizowany zbiór']
display_names = ['Pełny zbiór (111 cech)', 'Zoptymalizowany zbiór (73 cechy)']

for idx, dataset_name in enumerate(dataset_names):
    model = results_for_datasets[dataset_name]['model']
    X_train = results_for_datasets[dataset_name]['X_train']
    y_train = results_for_datasets[dataset_name]['y_train']

    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train, y_train,
        cv=5,
        scoring='roc_auc',
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10),
        random_state=42
    )

    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)

    axes[idx].plot(train_sizes, train_mean, label='Wynik uczenia', color='#1f77b4', linewidth=2, marker='o')
    axes[idx].fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color='#1f77b4')

    axes[idx].plot(train_sizes, val_mean, label='Wynik testowania', color='red', linewidth=2, marker='s', linestyle='--')
    axes[idx].fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2, color='red')

    axes[idx].set_xlabel('Liczba próbek uczących', fontsize=12)
    axes[idx].set_ylabel('Wynik ROC-AUC (skuteczność modelu)', fontsize=12)
    axes[idx].set_title(f'{display_names[idx]}', fontweight='bold', fontsize=14)
    axes[idx].legend(loc='lower right', fontsize=11)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim([0.985, 1.001])

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/xgboost_learning_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Krzywe uczenia zapisane")

## 7.3. Wpływ liczby estymatorów (n_estimators) na skuteczność modelu


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Wpływ liczby estymatorów (n_estimators) na skuteczność modelu', fontsize=16, fontweight='bold')

dataset_names_internal = ['Pełny zbiór (111 cech)', 'Zoptymalizowany zbiór (73 cechy)']
dataset_keys = ['Pełny zbiór', 'Zoptymalizowany zbiór']
param_range = np.array([10, 25, 50, 75, 100, 150, 200, 250, 300, 400, 500])

for idx, dataset_key in enumerate(dataset_keys):
    X_train_data = results_for_datasets[dataset_key]['X_train']
    y_train = results_for_datasets[dataset_key]['y_train']

    model_base = XGBClassifier(random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='logloss')

    train_scores, val_scores = validation_curve(
        model_base, X_train_data, y_train,
        param_name='n_estimators',
        param_range=param_range,
        cv=5,
        scoring='roc_auc',
        n_jobs=-1
    )

    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)

    axes[idx].plot(param_range, train_mean, label='Wynik uczenia', color='#1f77b4', linewidth=2, marker='o')
    axes[idx].fill_between(param_range, train_mean - train_std, train_mean + train_std, alpha=0.2, color='#1f77b4')

    axes[idx].plot(param_range, val_mean, label='Wynik testowania', color='red', linewidth=2, marker='s', linestyle='--')
    axes[idx].fill_between(param_range, val_mean - val_std, val_mean + val_std, alpha=0.2, color='red')

    axes[idx].set_xlabel('Liczba estymatorów (n_estimators)', fontsize=12)
    axes[idx].set_ylabel('Wynik ROC-AUC (skuteczność modelu)', fontsize=12)
    axes[idx].set_title(f'{dataset_names_internal[idx]}', fontweight='bold', fontsize=14)
    axes[idx].legend(loc='lower right', fontsize=11)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim([0.985, 1.001])

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/xgboost_validation_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Krzywe walidacyjne zapisane")

## 7.5. Ważność cech w obu modelach


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Top 15 najważniejszych cech w modelach', fontsize=16, fontweight='bold')

dataset_names = ['Pełny zbiór', 'Zoptymalizowany zbiór']
dataset_display = ['Pełny zbiór (111 cech)', 'Zoptymalizowany zbiór (73 cechy)']
colors_palette = ['Blues_d', 'Oranges_d']

for idx, dataset_name in enumerate(dataset_names):
    model = results_for_datasets[dataset_name]['model']
    X = results_for_datasets[dataset_name]['X_train']

    feature_importance = pd.DataFrame({
        'Cecha': [get_readable_name(col) for col in X.columns],
        'Ważność': model.feature_importances_
    }).sort_values('Ważność', ascending=False)

    top_features = feature_importance.head(15)

    sns.barplot(x='Ważność', y='Cecha', data=top_features,
                palette=colors_palette[idx], ax=axes[idx])
    axes[idx].set_xlabel('Ważność cechy', fontsize=12)
    axes[idx].set_ylabel('Cecha', fontsize=12)
    axes[idx].set_title(f'{dataset_display[idx]}', fontweight='bold', fontsize=14)
    axes[idx].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/xgboost_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Wizualizacja ważności cech zapisana")

## 8. Zapis modeli i wyników


In [ ]:
os.makedirs('../data', exist_ok=True)

for dataset_name in ['Pełny zbiór', 'Zoptymalizowany zbiór']:
    print(f"\n{'='*60}")
    print(f"ZAPISYWANIE MODELU - {dataset_name.upper()}")
    print(f"{'='*60}")

    model = results_for_datasets[dataset_name]['model']
    best_params = results_for_datasets[dataset_name]['best_params']
    X = data_splits[dataset_name]['X']

    model_data = {
        'model': model,
        'scaler': None,
        'feature_names': X.columns.tolist(),
        'best_params': best_params,
        'metrics': {
            'accuracy': evaluation_results[dataset_name]['accuracy'],
            'precision': evaluation_results[dataset_name]['precision'],
            'recall': evaluation_results[dataset_name]['recall'],
            'f1': evaluation_results[dataset_name]['f1'],
            'auc_roc': evaluation_results[dataset_name]['auc_roc'],
            'cv_mean': evaluation_results[dataset_name]['cv_mean'],
            'cv_std': evaluation_results[dataset_name]['cv_std']
        }
    }

    file_suffix = 'full' if dataset_name == 'Pełny zbiór' else 'optimized'

    model_path = f'{MODELS_DIR}/xgboost_model_{file_suffix}_url_corpus.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump(model_data, f)
    print(f"✓ Model zapisany: {model_path}")

comparison_df.to_csv('../data/url_corpus/xgboost_comparison.csv', index=False)
print(f"\n✓ Porównanie zapisane: ../data/url_corpus/xgboost_comparison.csv")

print(f"\n{'='*80}")
print("PODSUMOWANIE KOŃCOWE")
print(f"{'='*80}")

for dataset_name in ['Pełny zbiór', 'Zoptymalizowany zbiór']:
    print(f"\n{dataset_name.upper()}:")
    print(f"  Liczba cech: {len(data_splits[dataset_name]['X'].columns)}")
    print(f"  Dokładność: {evaluation_results[dataset_name]['accuracy']:.4f}")
    print(f"  F1-Score: {evaluation_results[dataset_name]['f1']:.4f}")
    print(f"  AUC-ROC: {evaluation_results[dataset_name]['auc_roc']:.4f}")
    print(f"  Czas trenowania: {results_for_datasets[dataset_name]['training_time']:.2f}s")

print(f"\n✅ Wszystkie modele i wizualizacje zostały zapisane!")
print(f"Wizualizacje zapisane w: {PLOTS_DIR}/")
print(f"Modele zapisane w: {MODELS_DIR}/")
